# CPU and CUDA Backend Parity

Forward traces and material gradients are compared between repository-local
CPU and CUDA libraries for 2D and 3D models at orders 2, 4, and 8. The test is
skipped only when CUDA is unavailable, unless `DEEPGPR_REQUIRE_CUDA=1`.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "tests"]
candidates.extend(parent / "tests" for parent in cwd.parents)
NOTEBOOK_DIR = next(
    (path for path in candidates if (path / "verification_utils.py").is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("verification_utils.py was not found from the current directory.")
notebook_path = str(NOTEBOOK_DIR)
if notebook_path not in sys.path:
    sys.path.insert(0, notebook_path)

import verification_utils as vu

REPO_ROOT = vu.configure_local_import()
for module_name in tuple(sys.modules):
    if module_name == "DeepGPR" or module_name.startswith("DeepGPR."):
        del sys.modules[module_name]
import DeepGPR

LOADED_PACKAGE = vu.assert_local_deepgpr(DeepGPR, REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")
print(f"DeepGPR package: {LOADED_PACKAGE}")


In [ ]:
import os
import torch

torch.manual_seed(2026)
CPU = torch.device("cpu")
CUDA = vu.selected_cuda_device()
CHECKS = []
METADATA = vu.runtime_metadata(DeepGPR, CPU)
REQUIRE_CUDA = os.environ.get("DEEPGPR_REQUIRE_CUDA", "0") == "1"
parity_rows = []

if CUDA is None:
    vu.record_check(
        CHECKS,
        "required CUDA availability",
        not REQUIRE_CUDA,
        require_cuda=REQUIRE_CUDA,
        cuda_available=False,
    )
    vu.record_skip(
        CHECKS,
        "CPU/CUDA numerical parity matrix",
        "CUDA is not available on this machine.",
    )
    CUDA_METADATA = None
else:
    CUDA_METADATA = vu.runtime_metadata(DeepGPR, CUDA)
    vu.record_check(
        CHECKS,
        "repository-local CUDA backend is available",
        True,
        device=str(CUDA),
        library=CUDA_METADATA["native_library"],
        abi=CUDA_METADATA["native_abi"],
    )


In [ ]:
def parity_run(device, shape, nt, order, mode):
    is_3d = len(shape) == 3
    er = torch.full(shape, 4.0, device=device, requires_grad=True)
    se = torch.full(shape, 3.0e-4, device=device, requires_grad=True)
    if is_3d:
        source_location = torch.tensor([[[8, 5, 8]]], dtype=torch.int32, device=device)
        receiver_location = torch.tensor(
            [[[6, 5, 6], [10, 5, 10]]], dtype=torch.int32, device=device
        )
    else:
        source_location = torch.tensor([[[6, 10, 0]]], dtype=torch.int32, device=device)
        receiver_location = torch.tensor(
            [[[6, 14, 0], [6, 18, 0]]], dtype=torch.int32, device=device
        )
    source = DeepGPR.wavelet.ricker(3.5e8, nt, 2.0e-11, 3.0e-9).reshape(1, nt, 1).to(device)
    result = DeepGPR.compute(
        device=device,
        dx=0.02,
        dt=2.0e-11,
        source_amplitudes=source,
        source_location=source_location,
        receiver_location=receiver_location,
        er=er,
        se=se,
        pmlthick=3 if is_3d else 4,
        fdtd_order=order,
        mode=mode,
        model_gradient_sampling_interval=1,
        wavefield_storage_dtype=torch.float32,
    )
    loss = result[-1].square().mean()
    loss.backward()
    vu.assert_finite("backend parity", result[-1], er.grad, se.grad)
    return {
        "receiver": result[-1].detach().cpu(),
        "grad_er": er.grad.detach().cpu(),
        "grad_se": se.grad.detach().cpu(),
    }


In [ ]:
if CUDA is not None:
    cases = [
        {"name": "2D", "shape": (24, 30), "nt": 180, "mode": 2},
        {"name": "3D", "shape": (16, 16, 16), "nt": 100, "mode": 3},
    ]
    for case in cases:
        for order in (2, 4, 8):
            cpu_result = parity_run(CPU, case["shape"], case["nt"], order, case["mode"])
            cuda_result = parity_run(CUDA, case["shape"], case["nt"], order, case["mode"])
            row = {
                "case": case["name"],
                "order": order,
                "receiver_relative_l2": vu.relative_l2(
                    cuda_result["receiver"], cpu_result["receiver"]
                ),
                "er_gradient_relative_l2": vu.relative_l2(
                    cuda_result["grad_er"], cpu_result["grad_er"]
                ),
                "se_gradient_relative_l2": vu.relative_l2(
                    cuda_result["grad_se"], cpu_result["grad_se"]
                ),
                "er_gradient_cosine": vu.cosine_similarity(
                    cuda_result["grad_er"], cpu_result["grad_er"]
                ),
                "se_gradient_cosine": vu.cosine_similarity(
                    cuda_result["grad_se"], cpu_result["grad_se"]
                ),
            }
            parity_rows.append(row)
            vu.record_check(
                CHECKS,
                f"CPU/CUDA parity for {case['name']} order {order}",
                row["receiver_relative_l2"] < 2.0e-4
                and max(
                    row["er_gradient_relative_l2"],
                    row["se_gradient_relative_l2"],
                ) < 5.0e-3
                and min(
                    row["er_gradient_cosine"],
                    row["se_gradient_cosine"],
                ) > 0.999,
                **row,
                receiver_tolerance=2.0e-4,
                gradient_tolerance=5.0e-3,
                cosine_tolerance=0.999,
            )


In [ ]:
vu.save_report(
    "06_cpu_cuda_parity",
    CHECKS,
    METADATA,
    extra={"cuda_metadata": CUDA_METADATA, "parity_rows": parity_rows},
)
print(f"Completed {len(CHECKS)} checks, including optional checks.")
